In [ ]:
PROJECT_ID = "qwiklabs-gcp-01-5fc8b912fc6a"   # <-- set to your Day-2 project
DATASET    = "aero_alerts"
LOCATION   = "US"
GCS_URI    = "gs://labs.roitraining.com/data-to-ai-workshop/airports.csv"

import requests, time, json, subprocess
import pandas as pd
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

bq.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET}"), exists_ok=True)
print("dataset ready")

dataset ready


In [ ]:
# make a connection that lets bigquery call gemini, then give it permission
!bq mk --connection --location=US --connection_type=CLOUD_RESOURCE gemini_conn 2>/dev/null

info = json.loads(subprocess.run(
    ["bq", "show", "--format=prettyjson", "--connection", f"{PROJECT_ID}.US.gemini_conn"],
    capture_output=True, text=True).stdout)
sa = info["cloudResource"]["serviceAccountId"]
print("service account:", sa)

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{sa}" --role="roles/aiplatform.user" --condition=None

BigQuery error in mk operation: Already Exists: Connection
projects/301800548063/locations/us/connections/gemini_conn
service account: bqcx-301800548063-srw0@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Updated IAM policy for project [qwiklabs-gcp-01-5fc8b912fc6a].
bindings:
- members:
  - serviceAccount:service-301800548063@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-301800548063@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-301800548063@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:bqcx-301800548063-srw0@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:qwiklabs-gcp-01-5fc8b912fc6a@qwiklabs-gcp-01-5fc8b912fc6a.iam.gserviceaccount.com
  role: roles/bigquery.admin
- members:
  - serviceAccount:301800548063@

In [ ]:
airports_table = f"{PROJECT_ID}.{DATASET}.airports"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
bq.load_table_from_uri(GCS_URI, airports_table, job_config=job_config).result()
print("loaded:", bq.get_table(airports_table).num_rows, "airports")

loaded: 82893 airports


In [ ]:
# the NWS api only covers the US, so filter to large US airports
large = bq.query(f"""
SELECT name, iata_code, municipality AS city, iso_region AS region,
       latitude_deg AS lat, longitude_deg AS lng
FROM `{airports_table}`
WHERE type = 'large_airport' AND iso_country = 'US'
ORDER BY iata_code
""").to_dataframe()

print(len(large), "large US airports")
large.head()

71 large US airports


,name,iata_code,city,region,lat,lng
0,Albuquerque International Sunport,ABQ,Albuquerque,US-NM,35.039976,-106.608925
1,Joint Base Andrews,ADW,Camp Springs,US-MD,38.810799,-76.866997
2,Ted Stevens Anchorage International Airport,ANC,Anchorage,US-AK,61.179004,-149.992561
3,Hartsfield Jackson Atlanta International Airport,ATL,Atlanta,US-GA,33.636700,-84.428101
4,Austin Bergstrom International Airport,AUS,Austin,US-TX,30.197535,-97.662015


In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

HEADERS = {"User-Agent": "AeroAlerts demo (student@qwiklabs.net)"}

# a session that auto-retries on rate-limit / server errors, waiting longer each time
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True,
)))

def get_forecast(lat, lng):
    # step 1: point -> forecast url    step 2: fetch the forecast
    pt = session.get(f"https://api.weather.gov/points/{lat},{lng}", headers=HEADERS, timeout=30)
    pt.raise_for_status()
    fc = session.get(pt.json()["properties"]["forecast"], headers=HEADERS, timeout=30)
    fc.raise_for_status()
    p = fc.json()["properties"]["periods"][0]
    return f"{p['name']}: {p['detailedForecast']}"

rows, failed = [], []
for a in large.itertuples():
    forecast = None
    for attempt in range(3):                 # extra safety net on top of the session retries
        try:
            forecast = get_forecast(a.lat, a.lng)
            break
        except Exception as e:
            print(f"retry {a.iata_code} ({attempt+1}/3): {e}")
            time.sleep(2 * (attempt + 1))
    if forecast is None:
        failed.append(a.iata_code)
    rows.append({"iata": a.iata_code, "name": a.name, "city": a.city,
                 "region": a.region, "lat": a.lat, "lng": a.lng, "forecast": forecast})
    time.sleep(0.5)                           # gentle pacing between airports

print(f"got forecasts for {sum(r['forecast'] is not None for r in rows)} of {len(rows)} airports")
if failed:
    print("still failed:", failed)

got forecasts for 71 of 71 airports


In [ ]:
forecast_table = f"{PROJECT_ID}.{DATASET}.airport_forecasts"

bq.load_table_from_dataframe(
    pd.DataFrame(rows), forecast_table,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
).result()
print("saved forecasts ->", forecast_table)

saved forecasts -> qwiklabs-gcp-01-5fc8b912fc6a.aero_alerts.airport_forecasts


In [ ]:
bq.query(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.gemini`
REMOTE WITH CONNECTION `{PROJECT_ID}.US.gemini_conn`
OPTIONS (endpoint = 'gemini-2.5-flash')
""").result()
print("gemini model ready")

gemini model ready


In [ ]:
alerts_table = f"{PROJECT_ID}.{DATASET}.airport_alerts"

bq.query(f"""
CREATE OR REPLACE TABLE `{alerts_table}` AS
SELECT iata, name, city, region, lat, lng, forecast,
       ml_generate_text_llm_result AS alert
FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.{DATASET}.gemini`,
  (
    SELECT *,
      CONCAT(
        'You are an FAA Aero Alerts assistant. Write a short, friendly weather alert ',
        '(2-3 sentences) for travelers at ', name, ' (', iata, ') in ', city, '. ',
        'Base it only on this NWS forecast: ', forecast, '. ',
        'Name any hazard (wind, snow, storms, heat) and give one practical tip; ',
        'if the weather is calm, give a brief reassuring note.'
      ) AS prompt
    FROM `{forecast_table}`
    WHERE forecast IS NOT NULL
  ),
  STRUCT(0.3 AS temperature, 512 AS max_output_tokens, TRUE AS flatten_json_output)
)
""").result()
print("alerts generated ->", alerts_table)

alerts generated -> qwiklabs-gcp-01-5fc8b912fc6a.aero_alerts.airport_alerts


In [ ]:
bq.query(f"""
SELECT iata, city, alert
FROM `{alerts_table}`
ORDER BY iata
LIMIT 10
""").to_dataframe()

,iata,city,alert
0,ABQ,Albuquerque,Hello travelers at ABQ! There's a 40% chance o...
1,ADW,Camp Springs,"Good morning, travelers at Joint Base Andrews!..."
2,ANC,Anchorage,"Good evening, travelers at ANC! Expect a partl..."
3,ATL,Atlanta,"Good morning, ATL travelers! Expect a sunny da..."
4,AUS,Austin,"Heads up, Austin travelers! There's a 30% chan..."
5,BDL,Hartford,"Good morning, travelers at BDL! Today looks li..."
6,BNA,Nashville,"Good morning, Nashville travelers! Expect a be..."
7,BOS,Boston,"Good morning, travelers at Logan! Today promis..."
8,BUF,Buffalo,"Good morning, Buffalo travelers! Today looks l..."
9,BWI,Baltimore,"Good morning, travelers! Expect a beautiful da..."
